# QuantLib Benchmark: SOFR + TermSOFR3m + CLP ICP Curves & Swap Sensitivities

Replicates the `QuantSupport` sensitivity example using QuantLib-Python.

**Key conventions (matching quantsupport):**
- Reference date: 2025-11-11
- Settlement: T+2 calendar days → 2025-11-13 (NullCalendar)
- Log-linear interpolation on discount factors (flat-forward)
- **NullCalendar** / **Unadjusted** BDC everywhere
- **Backward** date generation, no end-of-month
- Floating leg uses **simple forward rates** per coupon period (NOT daily-compounded OIS)
- Fixed leg: Semiannual, Floating leg: Quarterly
- Day count: Act/360, Compounding: Simple
- Bootstrap instruments start at reference_date (not T+2)
- DV01 via central-difference bump ±1bp

In [97]:
import QuantLib as ql
import pandas as pd
import numpy as np
import json
import os
import importlib

import data_loading, display, ql_helpers
importlib.reload(data_loading); importlib.reload(display); importlib.reload(ql_helpers)

from data_loading import load_rust_results, qs_sens_dict, qs_npv, qs_cashflows_df, qs_curve_df, qs_sens_full
from display import display_dv01, compare_discount_factors, display_cashflows
from ql_helpers import make_schedule, make_vanilla_swap, compute_dv01, build_collateral_curve, compute_xccy_npv

## 1. Global Settings

In [98]:
reference_date = ql.Date(11, 11, 2025)
ql.Settings.instance().evaluationDate = reference_date

# quantsupport uses NullCalendar + Unadjusted for everything
calendar = ql.NullCalendar()
day_count = ql.Actual360()

# T+2 settlement (calendar days, no holiday adjustment)
start_date = reference_date + 2  # 2025-11-13
notional = 10_000_000.0

print(f"Reference date : {reference_date}")
print(f"Start date     : {start_date}")

Reference date : November 11th, 2025
Start date     : November 13th, 2025


## 1b. Load Rust (quantsupport) Results

The sensitivity example outputs a JSON file with NPV, sensitivities (raw exposure + DV01),
cashflow details, and curve discount factors.  We load it here and use it throughout the
notebook instead of hard-coded values.

In [99]:
rust_products, rust_curves = load_rust_results()

print(f"Loaded Rust results: {len(rust_products)} products, {len(rust_curves)} curves")
for label, prod in rust_products.items():
    print(f"  - {label}: NPV = {prod['npv']:.2f}, {len(prod['sensitivities'])} sensitivities, {len(prod['cashflows'])} cashflows")

Loaded Rust results: 4 products, 4 curves
  - SOFR OIS 5Y Swap: NPV = 342.60, 13 sensitivities, 34 cashflows
  - TermSOFR3m 5Y Swap: NPV = 88754.68, 25 sensitivities, 34 cashflows
  - CLP ICP OIS 5Y Swap (USD collateral): NPV = 288.27, 34 sensitivities, 34 cashflows
  - Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR): NPV = 156003.53, 34 sensitivities, 44 cashflows


## 2. Market Data

All quotes from `examples/sensitivity/data/quotes.json`.

In [100]:
# SOFR OIS curve quotes
sofr_quotes = {
    "FixedRateDeposit_USD_SOFR_1D": 0.04330,
    "OIS_USD_SOFR_3M":  0.04335,
    "OIS_USD_SOFR_6M":  0.04250,
    "OIS_USD_SOFR_1Y":  0.04100,
    "OIS_USD_SOFR_2Y":  0.03920,
    "OIS_USD_SOFR_3Y":  0.03840,
    "OIS_USD_SOFR_4Y":  0.03800,
    "OIS_USD_SOFR_5Y":  0.03780,
    "OIS_USD_SOFR_7Y":  0.03770,
    "OIS_USD_SOFR_10Y": 0.03790,
    "OIS_USD_SOFR_15Y": 0.03850,
    "OIS_USD_SOFR_20Y": 0.03880,
    "OIS_USD_SOFR_30Y": 0.03750,
}

# TermSOFR3m curve quotes (basis spreads over SOFR)
termsofr_quotes = {
    "FixedRateDeposit_USD_TermSOFR3m_3M": 0.04365,
    "BasisSwap_USD_SOFR_TermSOFR3m_6M":  0.000127237986,
    "BasisSwap_USD_SOFR_TermSOFR3m_1Y":  0.000160745572,
    "BasisSwap_USD_SOFR_TermSOFR3m_2Y":  0.000196298404,
    "BasisSwap_USD_SOFR_TermSOFR3m_3Y":  0.000224136825,
    "BasisSwap_USD_SOFR_TermSOFR3m_4Y":  0.000243421966,
    "BasisSwap_USD_SOFR_TermSOFR3m_5Y":  0.000262792898,
    "BasisSwap_USD_SOFR_TermSOFR3m_7Y":  0.000298893256,
    "BasisSwap_USD_SOFR_TermSOFR3m_10Y": 0.000335114864,
    "BasisSwap_USD_SOFR_TermSOFR3m_15Y": 0.000378100224,
    "BasisSwap_USD_SOFR_TermSOFR3m_20Y": 0.000415419903,
    "BasisSwap_USD_SOFR_TermSOFR3m_30Y": 0.000465137190,
}

# CLP ICP curve quotes
icp_quotes = {
    "FixedRateDeposit_CLP_ICP_1D":  0.0500,
    "FixedRateDeposit_CLP_ICP_3M":  0.0498,
    "OIS_CLP_ICP_6M":  0.0495,
    "OIS_CLP_ICP_1Y":  0.0485,
    "OIS_CLP_ICP_2Y":  0.0470,
    "OIS_CLP_ICP_3Y":  0.0455,
    "OIS_CLP_ICP_5Y":  0.0440,
    "OIS_CLP_ICP_7Y":  0.0430,
    "OIS_CLP_ICP_10Y": 0.0425,
    "OIS_CLP_ICP_20Y": 0.0420,
}

# FX forward points (USD/CLP)
fx_fwd_quotes = {
    "FxForwardPoints_USDCLP_1M":  1.50,
    "FxForwardPoints_USDCLP_3M":  5.30,
    "FxForwardPoints_USDCLP_6M":  12.80,
}

# Cross-currency swap quotes (CLP fixed vs USD SOFR float)
xccy_quotes = {
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_1Y":  0.0510,
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_2Y":  0.0485,
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_3Y":  0.0465,
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_5Y":  0.0455,
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_7Y":  0.0450,
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_10Y": 0.0440,
    "CrossCurrencySwap_CLP_ICP_SOFR_USD_20Y": 0.0430,
}

# FX spot
fx_spot = 935.0
fx_spot_sq = ql.SimpleQuote(fx_spot)

# Merge all quotes into a single handle store for bumping
all_quote_handles = {}
for quotes_dict in [sofr_quotes, termsofr_quotes, icp_quotes, fx_fwd_quotes, xccy_quotes]:
    for name, rate in quotes_dict.items():
        all_quote_handles[name] = ql.SimpleQuote(rate)

def make_handle(name):
    return ql.QuoteHandle(all_quote_handles[name])

print(f"Total quotes: {len(all_quote_handles)}")

Total quotes: 45


## 3. Helper Functions

Build swaps matching quantsupport conventions: simple forward rates (VanillaSwap),
NullCalendar, Unadjusted, Backward date generation.

In [101]:
def make_handle(name):
    return ql.QuoteHandle(all_quote_handles[name])

## 4. Bootstrap the SOFR OIS Curve

quantsupport bootstraps OIS swaps with:
- Fixed Semi / Float Quarterly, both starting at reference_date
- Simple forward rates (not daily-compounded OIS)
- NPV = 0 as the bootstrap objective

We use `SwapRateHelper` with a quarterly `IborIndex` (simple forwards), not `OISRateHelper`.

In [102]:
sofr_helpers = []

# 1D deposit
sofr_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_USD_SOFR_1D"),
        ql.Period(1, ql.Days),
        0,                    # fixing days = 0 (starts at ref date)
        calendar,             # NullCalendar
        ql.Unadjusted,
        False,
        day_count,
    )
)

# IborIndex for projecting simple forwards in bootstrap
sofr_ibor_for_bootstrap = ql.IborIndex(
    "SimpleSOFR", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
)

ois_tenors = [
    ("OIS_USD_SOFR_3M",  ql.Period(3, ql.Months)),
    ("OIS_USD_SOFR_6M",  ql.Period(6, ql.Months)),
    ("OIS_USD_SOFR_1Y",  ql.Period(1, ql.Years)),
    ("OIS_USD_SOFR_2Y",  ql.Period(2, ql.Years)),
    ("OIS_USD_SOFR_3Y",  ql.Period(3, ql.Years)),
    ("OIS_USD_SOFR_4Y",  ql.Period(4, ql.Years)),
    ("OIS_USD_SOFR_5Y",  ql.Period(5, ql.Years)),
    ("OIS_USD_SOFR_7Y",  ql.Period(7, ql.Years)),
    ("OIS_USD_SOFR_10Y", ql.Period(10, ql.Years)),
    ("OIS_USD_SOFR_15Y", ql.Period(15, ql.Years)),
    ("OIS_USD_SOFR_20Y", ql.Period(20, ql.Years)),
    ("OIS_USD_SOFR_30Y", ql.Period(30, ql.Years)),
]

for name, tenor in ois_tenors:
    sofr_helpers.append(
        ql.SwapRateHelper(
            make_handle(name),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            sofr_ibor_for_bootstrap,
        )
    )

sofr_curve = ql.PiecewiseLogLinearDiscount(reference_date, sofr_helpers, day_count)
sofr_curve.enableExtrapolation()
sofr_curve_handle = ql.YieldTermStructureHandle(sofr_curve)

print("SOFR curve bootstrapped.")
for dt, df in sofr_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

SOFR curve bootstrapped.
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  November 12th, 2025  DF=0.9998797367  YF=0.002778
  February 11th, 2026  DF=0.9890430514  YF=0.255556
  May 11th, 2026  DF=0.9790789858  YF=0.502778
  November 11th, 2026  DF=0.9597061980  YF=1.013889
  November 11th, 2027  DF=0.9243882594  YF=2.027778
  November 11th, 2028  DF=0.8908288784  YF=3.044444
  November 11th, 2029  DF=0.8585791637  YF=4.058333
  November 11th, 2030  DF=0.8273197252  YF=5.072222
  November 11th, 2032  DF=0.7673403029  YF=7.102778
  November 11th, 2035  DF=0.6833842370  YF=10.144444
  November 11th, 2040  DF=0.5586557094  YF=15.219444
  November 11th, 2045  DF=0.4566471726  YF=20.291667
  November 11th, 2055  DF=0.3281415350  YF=30.436111


### SOFR Curve — Discount Factor Comparison

In [103]:
compare_discount_factors(sofr_curve, "SOFR", rust_curves, reference_date, day_count, "SOFR OIS Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: SOFR OIS Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998797367     0.9998797367       1.11e-16
  2025-12-11     0.0833     0.9964134592     0.9964134592       1.11e-16
  2026-02-11     0.2556     0.9890430514     0.9890430514       1.11e-16
  2026-05-11     0.5028     0.9790789858     0.9790789858      -6.88e-15
  2026-11-11     1.0139     0.9597061980     0.9597061980      -7.57e-14
  2027-11-11     2.0278     0.9243882594     0.9243882594       2.00e-15
  2028-11-11     3.0444     0.8908288784     0.8908288784      -4.76e-14
  2029-11-11     4.0583     0.8585791637     0.8585791637       2.22e-15
  2030-11-11     5.0722     0.8273197252     0.8273197252       

## 5. Price SOFR 5Y Swap & Sensitivities

In [104]:
maturity_5y = start_date + ql.Period(5, ql.Years)
print(f"Swap: {start_date} → {maturity_5y}, fixed = 3.78%, notional = {notional:,.0f}")

sofr_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.0378, notional,
    sofr_curve_handle, sofr_curve_handle,
    calendar, day_count,
)

# Read quantsupport results from JSON
rs_sofr_dv01 = qs_sens_full(rust_products, "SOFR OIS 5Y Swap")
rs_sofr_npv = qs_npv(rust_products, "SOFR OIS 5Y Swap")

sofr_handles = {k: all_quote_handles[k] for k in sofr_quotes}
sofr_npv, sofr_dv01 = compute_dv01(sofr_swap, sofr_handles)
display_dv01("SOFR OIS 5Y Swap", sofr_npv, sofr_dv01, rs_sofr_dv01, rs_npv=rs_sofr_npv)

Swap: November 13th, 2025 → November 13th, 2030, fixed = 3.78%, notional = 10,000,000
═══════════════════════════════════════════════════════════════════════════════════════════════
  SOFR OIS 5Y Swap
  NPV =         342.60 USD  (quantsupport:         342.60 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_USD_SOFR_1D                        2.75       2.75     27462.6312       0.00
  OIS_USD_SOFR_3M                                     2.78       2.78     27768.6580      -0.00
  OIS_USD_SOFR_6M                                     0.10       0.10      1014.6112       0.00
  OIS_USD_SOFR_1Y                                     0.00       0.00        30.0393       0.00
  OIS_USD_SOFR_2Y                                    -0.00      -0

### SOFR OIS 5Y Swap — Cashflow Details (quantsupport)

In [105]:
display_cashflows(rust_products, "SOFR OIS 5Y Swap")

payment_date      cashflow_type    side currency          notional rate_pct  year_fraction         amount  discount_factor
  2026-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.502778 190,050.000000         0.978866
  2026-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.511111 193,200.000000         0.959509
  2027-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.502778 190,050.000000         0.941833
  2027-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.511111 193,200.000000         0.924201
  2028-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.505556 191,100.000000         0.907362
  2028-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.511111 193,200.000000         0.890649
  2029-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  3.7800%       0.502778 190,050.000000         0.874511
  2029-11-13    

## 6. Bootstrap TermSOFR3m Curve

The TermSOFR3m curve is built from:
- A 3M deposit (outright rate)
- Basis swaps: pay SOFR + spread, receive TermSOFR3m Q → NPV = 0

The basis spread quote means `TermSOFR3m_par_swap_rate ≈ SOFR_par_swap_rate + spread`.

In [106]:
termsofr_helpers = []

# 3M deposit
termsofr_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_USD_TermSOFR3m_3M"),
        ql.Period(3, ql.Months),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

# For each basis swap tenor, compute SOFR par rate + spread → TermSOFR3m par rate
sofr_ibor_linked = ql.IborIndex(
    "SimpleSOFR", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
    sofr_curve_handle,
)

termsofr_ibor_for_bootstrap = ql.IborIndex(
    "TermSOFR3m", ql.Period(3, ql.Months),
    0, ql.USDCurrency(), calendar, ql.Unadjusted, False, day_count,
)
basis_tenors = [
    ("BasisSwap_USD_SOFR_TermSOFR3m_6M",  ql.Period(6, ql.Months)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_1Y",  ql.Period(1, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_2Y",  ql.Period(2, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_3Y",  ql.Period(3, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_4Y",  ql.Period(4, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_5Y",  ql.Period(5, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_7Y",  ql.Period(7, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_10Y", ql.Period(10, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_15Y", ql.Period(15, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_20Y", ql.Period(20, ql.Years)),
    ("BasisSwap_USD_SOFR_TermSOFR3m_30Y", ql.Period(30, ql.Years)),
]

# Store par rate SimpleQuote objects so bumping basis spreads propagates
# to the TermSOFR3m curve during DV01 computation.
termsofr_par_quotes = {}

# Annuity ratios and SOFR par swap helper data for refreshing par rates.
annuity_ratios = {}
sofr_par_swaps = {}          # tenor_name → temp_swap for recomputing sofr_par

for name, tenor in basis_tenors:
    maturity = reference_date + tenor
    fixed_sched = make_schedule(reference_date, maturity, ql.Period(ql.Semiannual), calendar)
    float_sched = make_schedule(reference_date, maturity, ql.Period(ql.Quarterly), calendar)

    # SOFR par-rate swap — kept alive so we can re-query fairRate() after a bump
    temp_swap = ql.VanillaSwap(
        ql.VanillaSwap.Receiver, 1.0,
        fixed_sched, 0.0, day_count,
        float_sched, sofr_ibor_linked, 0.0, day_count,
    )
    temp_swap.setPricingEngine(ql.DiscountingSwapEngine(sofr_curve_handle))
    sofr_par = temp_swap.fairRate()
    sofr_par_swaps[name] = temp_swap

    # Annuity ratio: floating-leg BPS / fixed-leg BPS
    ratio = abs(temp_swap.floatingLegBPS()) / abs(temp_swap.fixedLegBPS())
    annuity_ratios[name] = ratio

    basis_spread = all_quote_handles[name].value()
    termsofr_par = sofr_par + basis_spread * ratio

    par_sq = ql.SimpleQuote(termsofr_par)
    termsofr_par_quotes[name] = par_sq

    termsofr_helpers.append(
        ql.SwapRateHelper(
            ql.QuoteHandle(par_sq),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            termsofr_ibor_for_bootstrap,
            ql.QuoteHandle(),       # spread = 0
            ql.Period(),            # fwdStart = 0
            sofr_curve_handle,      # live SOFR discount — cross-curve effects flow through
        )
    )

termsofr_curve = ql.PiecewiseLogLinearDiscount(reference_date, termsofr_helpers, day_count)
termsofr_curve.enableExtrapolation()
termsofr_curve_handle = ql.YieldTermStructureHandle(termsofr_curve)

print("TermSOFR3m curve bootstrapped (with SOFR discount).")
for dt, df in termsofr_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

print()
print("Annuity ratios (A_qtr / A_semi):")
for name, r in annuity_ratios.items():
    print(f"  {name}: {r:.6f}")

TermSOFR3m curve bootstrapped (with SOFR discount).
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  February 11th, 2026  DF=0.9889680613  YF=0.255556
  May 11th, 2026  DF=0.9790174893  YF=0.502778
  November 11th, 2026  DF=0.9595515340  YF=1.013889
  November 11th, 2027  DF=0.9240227911  YF=2.027778
  November 11th, 2028  DF=0.8902229721  YF=3.044444
  November 11th, 2029  DF=0.8577316334  YF=4.058333
  November 11th, 2030  DF=0.8262142303  YF=5.072222
  November 11th, 2032  DF=0.7656959177  YF=7.102778
  November 11th, 2035  DF=0.6810216381  YF=10.144444
  November 11th, 2040  DF=0.5553456330  YF=15.219444
  November 11th, 2045  DF=0.4526128739  YF=20.291667
  November 11th, 2055  DF=0.3231446836  YF=30.436111

Annuity ratios (A_qtr / A_semi):
  BasisSwap_USD_SOFR_TermSOFR3m_6M: 1.005173
  BasisSwap_USD_SOFR_TermSOFR3m_1Y: 1.005097
  BasisSwap_USD_SOFR_TermSOFR3m_2Y: 1.004906
  BasisSwap_USD_SOFR_TermSOFR3m_3Y: 1.004822
  BasisSwap_USD_SOFR_TermSOFR3m_4Y: 1.004776
  BasisSwap_US

### TermSOFR3m Curve — Discount Factor Comparison

In [107]:
compare_discount_factors(termsofr_curve, "TermSOFR3m", rust_curves, reference_date, day_count, "TermSOFR3m Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: TermSOFR3m Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998794286     0.9998794286       0.00e+00
  2025-12-11     0.0833     0.9963891733     0.9963891733       1.11e-16
  2026-02-11     0.2556     0.9889680613     0.9889680613       1.11e-16
  2026-05-11     0.5028     0.9790174893     0.9790174893      -1.89e-15
  2026-11-11     1.0139     0.9595515340     0.9595515340       1.02e-13
  2027-11-11     2.0278     0.9240227911     0.9240227911      -1.74e-14
  2028-11-11     3.0444     0.8902229721     0.8902229721       7.80e-14
  2029-11-11     4.0583     0.8577316334     0.8577316334       5.15e-14
  2030-11-11     5.0722     0.8262142303     0.8262142303     

## 7. Price TermSOFR3m 5Y Swap & Sensitivities

Receive fixed 4.00% semi vs pay TermSOFR3m quarterly, 10MM USD.
Discount with SOFR (CSA = SOFR/USD).

In [108]:
termsofr_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.04, notional,
    termsofr_curve_handle, sofr_curve_handle,  # forecast=TermSOFR3m, discount=SOFR
    calendar, day_count,
    ibor_name="TermSOFR3m",
)

print(f"TermSOFR3m swap: {start_date} → {maturity_5y}, fixed = 4.00%")
print(f"NPV = {termsofr_swap.NPV():,.2f} USD")
print(f"Fair rate = {termsofr_swap.fairRate():.6%}")

TermSOFR3m swap: November 13th, 2025 → November 13th, 2030, fixed = 4.00%
NPV = 88,754.68 USD
Fair rate = 3.805675%


In [109]:
# Read quantsupport results from JSON
rs_termsofr_dv01 = qs_sens_full(rust_products, "TermSOFR3m 5Y Swap")
rs_termsofr_npv = qs_npv(rust_products, "TermSOFR3m 5Y Swap")

# TermSOFR3m DV01 with full cross-curve effects:
# When ANY quote (basis spread OR SOFR) is bumped, recompute
#   termsofr_par = new_sofr_par + spread × new_ratio
# so the TermSOFR3m curve re-bootstraps consistently.

def _refresh_par_rates(par_quotes, basis_handles, sofr_swaps, a_ratios):
    """Recompute termsofr_par = sofr_par + spread × ratio for every tenor."""
    for name, par_sq in par_quotes.items():
        sw = sofr_swaps[name]
        sofr_par = sw.fairRate()
        ratio = abs(sw.floatingLegBPS()) / abs(sw.fixedLegBPS())
        a_ratios[name] = ratio
        spread = basis_handles[name].value()
        par_sq.setValue(sofr_par + spread * ratio)

def compute_termsofr_dv01(swap, handles_to_bump, par_quotes,
                          basis_handles, sofr_swaps, a_ratios):
    bump = 1e-4
    _refresh_par_rates(par_quotes, basis_handles, sofr_swaps, a_ratios)
    base = swap.NPV()
    results = {}
    for name, sq in handles_to_bump.items():
        orig = sq.value()

        sq.setValue(orig + bump)
        _refresh_par_rates(par_quotes, basis_handles, sofr_swaps, a_ratios)
        up = swap.NPV()

        sq.setValue(orig - bump)
        _refresh_par_rates(par_quotes, basis_handles, sofr_swaps, a_ratios)
        dn = swap.NPV()

        sq.setValue(orig)
        _refresh_par_rates(par_quotes, basis_handles, sofr_swaps, a_ratios)

        results[name] = (up - dn) / 2.0
    return base, results

# Build dict of basis spread handles (subset of all_quote_handles)
basis_spread_handles = {name: all_quote_handles[name] for name, _ in basis_tenors}

combined_handles = {}
for k in termsofr_quotes:
    combined_handles[k] = all_quote_handles[k]
for k in sofr_quotes:
    combined_handles[k] = all_quote_handles[k]

termsofr_npv, termsofr_dv01 = compute_termsofr_dv01(
    termsofr_swap, combined_handles, termsofr_par_quotes,
    basis_spread_handles, sofr_par_swaps, annuity_ratios,
)
display_dv01("TermSOFR3m 5Y Swap", termsofr_npv, termsofr_dv01, rs_termsofr_dv01,
             rs_npv=rs_termsofr_npv)

═══════════════════════════════════════════════════════════════════════════════════════════════
  TermSOFR3m 5Y Swap
  NPV =       88754.68 USD  (quantsupport:       88754.68 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_USD_TermSOFR3m_3M                  5.56       5.56     55605.4795       0.00
  BasisSwap_USD_SOFR_TermSOFR3m_6M                   -0.01      -0.01      -111.6338       0.00
  BasisSwap_USD_SOFR_TermSOFR3m_1Y                    0.00       0.00        20.6891      -0.00
  BasisSwap_USD_SOFR_TermSOFR3m_2Y                   -0.00      -0.00       -42.7949      -0.00
  BasisSwap_USD_SOFR_TermSOFR3m_3Y                   -0.00      -0.00       -22.2111       0.00
  BasisSwap_USD_SOFR_TermSOFR3m_4Y                   -

### TermSOFR3m 5Y Swap — Cashflow Details (quantsupport)

In [110]:
display_cashflows(rust_products, "TermSOFR3m 5Y Swap")

payment_date      cashflow_type    side currency          notional rate_pct  year_fraction         amount  discount_factor
  2026-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.502778 201,111.111111         0.978866
  2026-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.511111 204,444.444444         0.959509
  2027-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.502778 201,111.111111         0.941833
  2027-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.511111 204,444.444444         0.924201
  2028-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.505556 202,222.222222         0.907362
  2028-11-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.511111 204,444.444444         0.890649
  2029-05-13    FixedRateCoupon Receive      USD 10,000,000.000000  4.0000%       0.502778 201,111.111111         0.874511
  2029-11-13    

## 8. Collateral(CLP,USD) Curve — Manual Bootstrap

The Collateral curve must be bootstrapped **before** the ICP curve because in
quantsupport the ICP OIS instruments (6M+) are discounted with
`Collateral(CLP,USD)`, not with ICP itself.

The Collateral bootstrap instruments are **fix-float cross-currency swaps**
(CLP fixed vs USD SOFR floating), so they depend only on SOFR — no circular
dependency with ICP.

**Bootstrap approach** (pure Python, no ORE):
- **Short end** (FX fwd points): `df_clp = df_sofr × spot / (spot + pts)`
- **Long end** (xccy swap rates): sequential 1-D root-finding with `scipy.optimize.brentq`

In [111]:
# ── Collateral(CLP,USD) bootstrap ──────────────────────────────────
fx_fwd_specs = [
    ("FxForwardPoints_USDCLP_1M", ql.Period(1, ql.Months)),
    ("FxForwardPoints_USDCLP_3M", ql.Period(3, ql.Months)),
    ("FxForwardPoints_USDCLP_6M", ql.Period(6, ql.Months)),
]
xccy_specs = [
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_1Y",  ql.Period(1,  ql.Years)),
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_2Y",  ql.Period(2,  ql.Years)),
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_3Y",  ql.Period(3,  ql.Years)),
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_5Y",  ql.Period(5,  ql.Years)),
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_7Y",  ql.Period(7,  ql.Years)),
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_10Y", ql.Period(10, ql.Years)),
    ("CrossCurrencySwap_CLP_ICP_SOFR_USD_20Y", ql.Period(20, ql.Years)),
]

collateral_curve = build_collateral_curve(
    reference_date, calendar, day_count,
    sofr_curve, fx_spot, fx_fwd_specs, xccy_specs, all_quote_handles)
collateral_handle = ql.YieldTermStructureHandle(collateral_curve)

print("Collateral(CLP,USD) curve nodes:")
print(f"{'Date':>30}  {'DF':>14}  {'YF':>8}")
print("-" * 56)
for dt, df in collateral_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"{str(dt):>30}  {df:>14.10f}  {yf:>8.4f}")

Collateral(CLP,USD) curve nodes:
                          Date              DF        YF
--------------------------------------------------------
           November 11th, 2025    1.0000000000    0.0000
           December 11th, 2025    0.9948174953    0.0833
           February 11th, 2026    0.9834683112    0.2556
                May 11th, 2026    0.9658565644    0.5028
           November 11th, 2026    0.9504585419    1.0139
           November 11th, 2027    0.9077313783    2.0278
           November 11th, 2028    0.8699669817    3.0444
           November 11th, 2030    0.7967644270    5.0722
           November 11th, 2032    0.7300453272    7.1028
           November 11th, 2035    0.6450865335   10.1444
           November 11th, 2045    0.4253319787   20.2917


In [112]:
compare_discount_factors(collateral_curve, "Collateral(CLP/USD)", rust_curves,
                        reference_date, day_count, "Collateral(CLP, USD) Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: Collateral(CLP, USD) Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998268156     0.9998268156       0.00e+00
  2025-12-11     0.0833     0.9948174953     0.9948174953       7.77e-16
  2026-02-11     0.2556     0.9834683112     0.9834683112       1.67e-15
  2026-05-11     0.5028     0.9658565644     0.9658565644       9.88e-15
  2026-11-11     1.0139     0.9504585419     0.9504585419      -7.77e-16
  2027-11-11     2.0278     0.9077313783     0.9077313783      -6.66e-16
  2028-11-11     3.0444     0.8699669817     0.8699669817      -4.44e-16
  2029-11-11     4.0583     0.8325615555     0.8325615555      -3.33e-16
  2030-11-11     5.0722     0.7967644270     0.79676

## 9. Bootstrap CLP ICP Curve (dual-curve: discount on Collateral)

Same conventions: NullCalendar, Unadjusted, Backward, Semi fixed / Quarterly float.

In quantsupport the ICP OIS instruments (6M+) are **discounted with
`Collateral(CLP,USD)`**, not with ICP itself. Only the short-end deposits
(1D, 3M) self-discount.  We pass `collateral_handle` to `SwapRateHelper`
via its `discountingCurve` parameter.

In [113]:
icp_helpers = []

# 1D deposit (self-discounting — matches Rust FixedIncome asset class)
icp_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_1D"),
        ql.Period(1, ql.Days),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

# 3M deposit (self-discounting)
icp_helpers.append(
    ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_3M"),
        ql.Period(3, ql.Months),
        0, calendar, ql.Unadjusted, False, day_count,
    )
)

icp_ibor_for_bootstrap = ql.IborIndex(
    "SimpleICP", ql.Period(3, ql.Months),
    0, ql.CLPCurrency(), calendar, ql.Unadjusted, False, day_count,
)

icp_tenors = [
    ("OIS_CLP_ICP_6M",  ql.Period(6, ql.Months)),
    ("OIS_CLP_ICP_1Y",  ql.Period(1, ql.Years)),
    ("OIS_CLP_ICP_2Y",  ql.Period(2, ql.Years)),
    ("OIS_CLP_ICP_3Y",  ql.Period(3, ql.Years)),
    ("OIS_CLP_ICP_5Y",  ql.Period(5, ql.Years)),
    ("OIS_CLP_ICP_7Y",  ql.Period(7, ql.Years)),
    ("OIS_CLP_ICP_10Y", ql.Period(10, ql.Years)),
    ("OIS_CLP_ICP_20Y", ql.Period(20, ql.Years)),
]

for name, tenor in icp_tenors:
    icp_helpers.append(
        ql.SwapRateHelper(
            make_handle(name),
            tenor,
            calendar,
            ql.Semiannual,
            ql.Unadjusted,
            day_count,
            icp_ibor_for_bootstrap,
            ql.QuoteHandle(),       # spread = 0
            ql.Period(),            # fwdStart = 0
            collateral_handle,      # discount on Collateral(CLP,USD)
        )
    )

icp_curve = ql.PiecewiseLogLinearDiscount(reference_date, icp_helpers, day_count)
icp_curve.enableExtrapolation()
icp_curve_handle = ql.YieldTermStructureHandle(icp_curve)

print("CLP ICP curve bootstrapped (discount = Collateral).")
for dt, df in icp_curve.nodes():
    yf = day_count.yearFraction(reference_date, dt)
    print(f"  {dt}  DF={df:.10f}  YF={yf:.6f}")

CLP ICP curve bootstrapped (discount = Collateral).
  November 11th, 2025  DF=1.0000000000  YF=0.000000
  November 12th, 2025  DF=0.9998611304  YF=0.002778
  February 11th, 2026  DF=0.9874332660  YF=0.255556
  May 11th, 2026  DF=0.9757932516  YF=0.502778
  November 11th, 2026  DF=0.9526192657  YF=1.013889
  November 11th, 2027  DF=0.9102091081  YF=2.027778
  November 11th, 2028  DF=0.8722857253  YF=3.044444
  November 11th, 2030  DF=0.8026219204  YF=5.072222
  November 11th, 2032  DF=0.7405112031  YF=7.102778
  November 11th, 2035  DF=0.6544305203  YF=10.144444
  November 11th, 2045  DF=0.4325233448  YF=20.291667


### CLP ICP Curve — Discount Factor Comparison

In [114]:
compare_discount_factors(icp_curve, "ICP", rust_curves, reference_date, day_count, "CLP ICP Curve")


────────────────────────────────────────────────────────────────────────────────
  Discount Factor Comparison: CLP ICP Curve
────────────────────────────────────────────────────────────────────────────────
  Date               YF            QL DF            QS DF           Diff
  --------------------------------------------------------------------
  2025-11-12     0.0028     0.9998611304     0.9998611304       0.00e+00
  2025-12-11     0.0833     0.9958837145     0.9958837145       0.00e+00
  2026-02-11     0.2556     0.9874332660     0.9874332660       0.00e+00
  2026-05-11     0.5028     0.9757932516     0.9757932516      -2.00e-15
  2026-11-11     1.0139     0.9526192657     0.9526192657      -5.36e-14
  2027-11-11     2.0278     0.9102091081     0.9102091081       1.33e-15
  2028-11-11     3.0444     0.8722857253     0.8722857253       1.33e-15
  2029-11-11     4.0583     0.8367291342     0.8367291342      -3.05e-14
  2030-11-11     5.0722     0.8026219204     0.8026219204      -5

## 10. Price CLP ICP 5Y Swap & Sensitivities

Receive fixed 4.40% semi vs pay ICP quarterly, 5B CLP.

**Discount curve**: CLP cashflows are discounted off `Collateral(CLP,USD)`,
matching quantsupport's CSA policy. NPVs converted to USD at FX spot = 935.

In [115]:
clp_notional = 5_000_000_000.0

icp_swap = make_vanilla_swap(
    start_date, maturity_5y, 0.0440, clp_notional,
    icp_curve_handle, collateral_handle,  # forecast=ICP, discount=Collateral
    calendar, day_count,
    ibor_name="SimpleICP", ccy=ql.CLPCurrency(),
)

icp_npv_clp = icp_swap.NPV()
icp_npv_usd = icp_npv_clp / fx_spot

print(f"CLP ICP swap: {start_date} → {maturity_5y}, fixed = 4.40%")
print(f"NPV = {icp_npv_clp:,.2f} CLP = {icp_npv_usd:,.2f} USD")
print(f"Fair rate = {icp_swap.fairRate():.6%}")

CLP ICP swap: November 13th, 2025 → November 13th, 2030, fixed = 4.40%
NPV = 269,535.98 CLP = 288.27 USD
Fair rate = 4.398793%


In [116]:
# Read quantsupport results from JSON
rs_icp_dv01 = qs_sens_full(rust_products, "CLP ICP OIS 5Y Swap (USD collateral)")
rs_icp_npv = qs_npv(rust_products, "CLP ICP OIS 5Y Swap (USD collateral)")

# The ICP swap discounts on Collateral, so we need handles for:
# 1. ICP quotes (affect ICP forward rates => forecast curve)
# 2. FX fwd, xccy, fx_spot (affect Collateral => discount curve)
# 3. SOFR quotes (affect Collateral indirectly via SOFR curve)
icp_all_handles = {}
for k in icp_quotes:
    icp_all_handles[k] = all_quote_handles[k]
for name, _ in fx_fwd_specs:
    icp_all_handles[name] = all_quote_handles[name]
for name, _ in xccy_specs:
    icp_all_handles[name] = all_quote_handles[name]
icp_all_handles["USD/CLP"] = fx_spot_sq
for k in sofr_quotes:
    icp_all_handles[k] = all_quote_handles[k]

def icp_reprice():
    """Rebuild Collateral + re-bootstrap ICP, then reprice the ICP swap."""
    cc = build_collateral_curve(
        reference_date, calendar, day_count,
        sofr_curve, fx_spot_sq.value(), fx_fwd_specs, xccy_specs,
        all_quote_handles)
    cc_h = ql.YieldTermStructureHandle(cc)

    helpers = []
    helpers.append(ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_1D"),
        ql.Period(1, ql.Days), 0, calendar, ql.Unadjusted, False, day_count))
    helpers.append(ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_3M"),
        ql.Period(3, ql.Months), 0, calendar, ql.Unadjusted, False, day_count))
    ibor = ql.IborIndex("SimpleICP", ql.Period(3, ql.Months),
        0, ql.CLPCurrency(), calendar, ql.Unadjusted, False, day_count)
    for name, tenor in icp_tenors:
        helpers.append(ql.SwapRateHelper(
            make_handle(name), tenor, calendar, ql.Semiannual,
            ql.Unadjusted, day_count, ibor,
            ql.QuoteHandle(), ql.Period(), cc_h))
    ic = ql.PiecewiseLogLinearDiscount(reference_date, helpers, day_count)
    ic.enableExtrapolation()
    ic_h = ql.YieldTermStructureHandle(ic)

    swap = make_vanilla_swap(
        start_date, maturity_5y, 0.0440, clp_notional,
        ic_h, cc_h, calendar, day_count,
        ibor_name="SimpleICP", ccy=ql.CLPCurrency())
    return swap.NPV() / fx_spot_sq.value()

bump = 1e-4
icp_npv_usd_base = icp_reprice()
icp_dv01 = {}
for name, sq in icp_all_handles.items():
    orig = sq.value()
    sq.setValue(orig + bump)
    up = icp_reprice()
    sq.setValue(orig - bump)
    dn = icp_reprice()
    sq.setValue(orig)
    d = (up - dn) / 2.0
    if abs(d) > 0.005:
        icp_dv01[name] = d

display_dv01("CLP ICP OIS 5Y Swap (USD equiv)", icp_npv_usd_base, icp_dv01, rs_icp_dv01,
             rs_npv=rs_icp_npv)

═══════════════════════════════════════════════════════════════════════════════════════════════
  CLP ICP OIS 5Y Swap (USD equiv)
  NPV =         288.27 USD  (quantsupport:         288.27 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_CLP_ICP_1D                         1.46       1.46     14624.0843       0.00
  FixedRateDeposit_CLP_ICP_3M                         1.56       1.56     15647.9710       0.00
  OIS_CLP_ICP_6M                                     -0.07      -0.07      -745.1355      -0.00
  OIS_CLP_ICP_1Y                                      0.03       0.03       257.7173      -0.00
  OIS_CLP_ICP_5Y                                  -2381.76   -2381.76 -23817571.6476      -0.00
  OIS_CLP_ICP_7Y                         

### CLP ICP OIS 5Y Swap — Cashflow Details (quantsupport)

In [117]:
display_cashflows(rust_products, "CLP ICP OIS 5Y Swap (USD collateral)")

payment_date      cashflow_type    side currency             notional rate_pct  year_fraction             amount  discount_factor
  2026-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.502778 110,611,111.111111         0.965688
  2026-11-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.511111 112,444,444.444444         0.950219
  2027-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.502778 110,611,111.111111         0.928791
  2027-11-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.511111 112,444,444.444444         0.907521
  2028-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.505556 111,222,222.222222         0.888545
  2028-11-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.511111 112,444,444.444444         0.869758
  2029-05-13    FixedRateCoupon Receive      CLP 5,000,000,000.000000  4.4000%       0.502

## 11. Cross-Currency Swap CLP/USD 5Y & Sensitivities

**Trade**: receive CLP ICP + 50 bp, pay USD SOFR, 5Y, quarterly both legs.
CLP notional = 10 M USD × 935 = 9.35 B CLP, USD notional = 10 M.

Discounting: CLP leg with Collateral(CLP,USD) + FX conversion, USD leg with SOFR.

In [118]:
# ── XCcy swap NPV ─────────────────────────────────────────────────
usd_notional = notional  # 10 M
clp_notional_xccy = usd_notional * fx_spot  # 9.35 B

schedule_q = make_schedule(start_date, maturity_5y, ql.Period(ql.Quarterly), calendar)

xccy_npv = compute_xccy_npv(
    sofr_curve, icp_curve, collateral_curve,
    schedule_q, clp_notional_xccy, usd_notional, 0.005,
    fx_spot, day_count, start_date, maturity_5y)

rs_xccy_npv = qs_npv(rust_products,
    "Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)")

print(f"XCcy swap: {start_date} → {maturity_5y}, ICP+50bp vs SOFR, CLP notional = {clp_notional_xccy:,.0f}")
print(f"  QL NPV  = {xccy_npv:>12,.2f} USD")
print(f"  Rust NPV= {rs_xccy_npv:>12,.2f} USD")
print(f"  Diff    = {xccy_npv - rs_xccy_npv:>12,.2f} USD")

XCcy swap: November 13th, 2025 → November 13th, 2030, ICP+50bp vs SOFR, CLP notional = 9,350,000,000
  QL NPV  =   156,003.53 USD
  Rust NPV=   156,003.53 USD
  Diff    =         0.00 USD


In [119]:
# ── XCcy DV01 (central-difference, 1 bp bump) ────────────────────
bump = 1e-4

# Collect all handles that may affect the xccy NPV
xccy_handles = {}
for k, sq in sofr_handles.items():
    xccy_handles[k] = sq
for k, sq in icp_all_handles.items():
    xccy_handles[k] = sq
for name, _ in fx_fwd_specs:
    xccy_handles[name] = all_quote_handles[name]
for name, _ in xccy_specs:
    xccy_handles[name] = all_quote_handles[name]
xccy_handles["USD/CLP"] = fx_spot_sq

def rebuild_icp(cc_h):
    """Re-bootstrap ICP with the given Collateral handle."""
    helpers = []
    helpers.append(ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_1D"),
        ql.Period(1, ql.Days), 0, calendar, ql.Unadjusted, False, day_count))
    helpers.append(ql.DepositRateHelper(
        make_handle("FixedRateDeposit_CLP_ICP_3M"),
        ql.Period(3, ql.Months), 0, calendar, ql.Unadjusted, False, day_count))
    ibor = ql.IborIndex("SimpleICP", ql.Period(3, ql.Months),
        0, ql.CLPCurrency(), calendar, ql.Unadjusted, False, day_count)
    for nm, tn in icp_tenors:
        helpers.append(ql.SwapRateHelper(
            make_handle(nm), tn, calendar, ql.Semiannual,
            ql.Unadjusted, day_count, ibor,
            ql.QuoteHandle(), ql.Period(), cc_h))
    ic = ql.PiecewiseLogLinearDiscount(reference_date, helpers, day_count)
    ic.enableExtrapolation()
    return ic

def xccy_reprice():
    """Rebuild Collateral + ICP + xccy NPV from scratch."""
    cc = build_collateral_curve(
        reference_date, calendar, day_count,
        sofr_curve, fx_spot_sq.value(), fx_fwd_specs, xccy_specs,
        all_quote_handles)
    cc_h = ql.YieldTermStructureHandle(cc)
    ic = rebuild_icp(cc_h)
    return compute_xccy_npv(
        sofr_curve, ic, cc,
        schedule_q, clp_notional_xccy, usd_notional, 0.005,
        fx_spot_sq.value(), day_count, start_date, maturity_5y)

xccy_dv01 = {}
for name, sq in xccy_handles.items():
    orig = sq.value()
    sq.setValue(orig + bump)
    up = xccy_reprice()
    sq.setValue(orig - bump)
    dn = xccy_reprice()
    sq.setValue(orig)
    xcxy_d = (up - dn) / 2.0
    if abs(xcxy_d) > 0.005:
        xccy_dv01[name] = xcxy_d

rs_xccy_dv01 = qs_sens_full(rust_products,
    "Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)")

display_dv01("XCcy Swap CLP/USD 5Y", xccy_npv, xccy_dv01,
             rs_xccy_dv01, rs_npv=rs_xccy_npv)

═══════════════════════════════════════════════════════════════════════════════════════════════
  XCcy Swap CLP/USD 5Y
  NPV =      156003.53 USD  (quantsupport:      156003.53 USD)
═══════════════════════════════════════════════════════════════════════════════════════════════
  Pillar                                           QL DV01    QS DV01    QS Exposure       Diff
  -------------------------------------------------------------------------------------------
  FixedRateDeposit_USD_SOFR_1D                        5.37       5.37     53664.1498       0.00
  OIS_USD_SOFR_3M                                    -0.05      -0.05      -491.3090       0.00
  OIS_USD_SOFR_6M                                    -0.39      -0.39     -3875.1887       0.00
  FixedRateDeposit_CLP_ICP_1D                        -2.73      -2.73    -27347.0376      -0.00
  FixedRateDeposit_CLP_ICP_3M                        -2.93      -2.93    -29261.7058      -0.00
  OIS_CLP_ICP_6M                                    

### XCcy Swap — Cashflow Details (quantsupport)

In [120]:
display_cashflows(rust_products,
    "Cross-Currency Swap CLP/USD 5Y (receive CLP ICP, pay USD SOFR)")

payment_date      cashflow_type    side currency             notional rate_pct  year_fraction             amount  discount_factor
  2026-02-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.9761%       0.255556 130,848,304.631490         0.983069
  2026-05-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.8230%       0.247222 123,042,245.486662         0.965688
  2026-08-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.7310%       0.255556 124,990,708.008392         0.957959
  2026-11-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.7263%       0.255556 124,879,848.136303         0.950219
  2027-02-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.5176%       0.255556 119,892,497.843887         0.939266
  2027-05-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.5167%       0.247222 115,963,381.248170         0.928791
  2027-08-13 FloatingRateCoupon Receive      CLP 9,350,000,000.000000  4.5176%       0.255

## 12. Summary

In [121]:
print("NPV Comparison Summary")
print("=" * 70)
print(f"{'Product':<40} {'QuantLib':>12} {'quantsupport':>12}")
print("-" * 70)
print(f"{'SOFR OIS 5Y Swap (USD)':<40} {sofr_npv:>12.2f} {qs_npv(rust_products, 'SOFR OIS 5Y Swap'):>12.2f}")
print(f"{'TermSOFR3m 5Y Swap (USD)':<40} {termsofr_npv:>12.2f} {qs_npv(rust_products, 'TermSOFR3m 5Y Swap'):>12.2f}")
print(f"{'CLP ICP 5Y Swap (USD equiv)':<40} {icp_npv_usd_base:>12.2f} {qs_npv(rust_products, 'CLP ICP OIS 5Y Swap (USD collateral)'):>12.2f}")
print(f"{'XCcy Swap CLP/USD 5Y (USD)':<40} {xccy_npv:>12.2f} {rs_xccy_npv:>12.2f}")
print("=" * 70)
print()
print("Convention notes:")
print("  - VanillaSwap with IborIndex (simple fwd rates), not OvernightIndexedSwap")
print("  - NullCalendar, Unadjusted BDC, Backward date generation")
print("  - Log-linear on DFs (flat forward between nodes)")
print("  - TermSOFR3m: static par-rate approach for basis curve")
print("  - Collateral(CLP,USD): manual bootstrap (scipy.optimize.brentq)")
print("  - ICP curve: dual-curve bootstrap, OIS instruments discount on Collateral")
print("  - XCcy NPV: manual cashflow PV computation (no QL instrument)")

NPV Comparison Summary
Product                                      QuantLib quantsupport
----------------------------------------------------------------------
SOFR OIS 5Y Swap (USD)                         342.60       342.60
TermSOFR3m 5Y Swap (USD)                     88754.68     88754.68
CLP ICP 5Y Swap (USD equiv)                    288.27       288.27
XCcy Swap CLP/USD 5Y (USD)                  156003.53    156003.53

Convention notes:
  - VanillaSwap with IborIndex (simple fwd rates), not OvernightIndexedSwap
  - NullCalendar, Unadjusted BDC, Backward date generation
  - Log-linear on DFs (flat forward between nodes)
  - TermSOFR3m: static par-rate approach for basis curve
  - Collateral(CLP,USD): manual bootstrap (scipy.optimize.brentq)
  - ICP curve: dual-curve bootstrap, OIS instruments discount on Collateral
  - XCcy NPV: manual cashflow PV computation (no QL instrument)
